In [ ]:
"""
exp06 Step 01: Capped-Regret Re-Entry Exploration (EXPLORATION ON 2005-2014 ONLY; context, PROTOCOL.md §9)

The 12 candidates of PROTOCOL.md §5 (exit on the trend signal or near the all-time high; buy back at once if the close
rises b above the sale, with a cooling-off of c decisions, or on a recovery above the 200-session average after a close
below it) are replayed as ONE continuous path over 2005-01-03 -> 2014-12-31 (start invested, no forced buy-back), against
buy-and-hold over the same sessions. The data is cut at so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18).

What it shows: how often each exit is wrong (ended by the buy-stop) and the roadmap arithmetic (1 - q) x G vs q x |L|.
DATA-DEPENDENT FOLLOW-UP of exp03; 2008 is known to everyone (hindsight-prone). The result does not change the grid or
the selection (pre-registered). Outputs: store04_experiments/exp06_capped_regret_reentry/step01_exploration_data/.
Trials: 12 (stage "exploration").
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp06_capped_regret_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE CONTINUOUS REPLAY FUNCTIONS
from so.core.continuous_replay import simulate_continuous_replay_dict, get_episode_scorecard_pdf, get_replay_summary_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp06_capped_regret_reentry.rules import get_rule_candidate_list, get_rule_builder_func, get_trigger_summary_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

In [ ]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=config.EXPLORATION_DATA_CUTOFF_DATE_STR)
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DEFINE THE EXPLORATION WINDOW
exploration_start_str, exploration_end_str = config.EXPLORATION_START_DATE_STR, config.EXPLORATION_END_DATE_STR
# DEFINE ONE PERIOD PER CALENDAR YEAR (FOR THE YEARS-WON COUNT)
session_date_arr = np.array(ohlcv_array_dict["session_date_list"])
window_date_arr = session_date_arr[(session_date_arr >= pd.Timestamp(exploration_start_str).date()) & (session_date_arr <= pd.Timestamp(exploration_end_str).date())]
year_period_pdf = pd.DataFrame([{"period_id": year, "period_start": min(d for d in window_date_arr if d.year == year), "period_end": max(d for d in window_date_arr if d.year == year)}
                                for year in sorted({d.year for d in window_date_arr})])
# DISPLAY INFORMATION
print(f"Data:\t{daily_pdf['date'].min()} -> {daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions)\tReplay:\t{exploration_start_str} -> {exploration_end_str}\t({len(window_date_arr):,} sessions, {len(year_period_pdf)} years)")

In [ ]:
"""
The 12 candidates, each one continuous path (start invested, no forced buy-back), against buy-and-hold.
"""
# SIMULATE BUY AND HOLD
buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)
print(f"Buy & hold:\ttotal return {buy_hold_dict['metric_dict']['total_return']:.2%} | Sharpe {buy_hold_dict['metric_dict']['sharpe_ratio']:.2f} | max drawdown {buy_hold_dict['metric_dict']['max_drawdown']:.2%}")
# DEFINE THE RULE BUILDER
rule_builder_func = get_rule_builder_func(daily_pdf)
# LISTS TO HOLD THE RESULTS
row_dict_list, scorecard_pdf_list = [], []
# ITERATE OVER THE CANDIDATES
for candidate_idx, candidate_dict in enumerate(get_rule_candidate_list()):
    # BUILD AND REPLAY THE CANDIDATE
    exit_signal_arr, reentry_func, reason_str = rule_builder_func(candidate_dict)
    simulation_dict = simulate_continuous_replay_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, exp_config.UNTOUCHED_START_DATE_STR,
                                                      volatility_arr_in=np.full(len(daily_pdf), np.nan), stop_k_in=None, reentry_func_in=reentry_func,
                                                      reentry_reason_str_in=reason_str, exit_signal_arr_in=exit_signal_arr, max_cash_sessions_in=exp_config.MAX_CASH_SESSIONS)
    # SUMMARIZE THE PATH (YEARS AS PERIODS) AND THE TRIGGERS
    summary_dict = get_replay_summary_dict(simulation_dict["daily_equity_pdf"], buy_hold_dict["daily_equity_pdf"], year_period_pdf, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
    trigger_dict = get_trigger_summary_dict(simulation_dict)
    scorecard_pdf = get_episode_scorecard_pdf(simulation_dict, ohlcv_array_dict)
    scorecard_pdf_list.append(scorecard_pdf.assign(candidate_idx=candidate_idx, **candidate_dict))
    # STORE THE ROW
    row_dict_list.append({"candidate_idx": candidate_idx, **candidate_dict, "total_return": summary_dict["strategy_total_return"], "buy_hold_total_return": summary_dict["buy_hold_total_return"],
                          "excess_total_return": summary_dict["excess_total_return"], "annualized_return": summary_dict["strategy_annualized_return"],
                          "sharpe_ratio": summary_dict["strategy_sharpe_ratio"], "max_drawdown": summary_dict["strategy_max_drawdown"], "in_market_pct": summary_dict["in_market_pct"],
                          "exit_count": simulation_dict["metric_dict"]["signal_exit_count"], "mean_cash_sessions": simulation_dict["metric_dict"]["mean_cash_sessions"],
                          "bought_back_lower_pct": simulation_dict["metric_dict"]["positive_episode_pct"], **trigger_dict,
                          "years_won": summary_dict["period_windows_beating_buy_hold"], "year_count": summary_dict["period_window_count"],
                          "annualized_log_excess": summary_dict["log_excess_annualized_log_excess"], "log_excess_ci_low": summary_dict["log_excess_ci_low"], "log_excess_ci_high": summary_dict["log_excess_ci_high"]})
# CONVERT THE RESULTS
exploration_pdf = pd.DataFrame(row_dict_list)
exploration_scorecard_pdf = pd.concat(scorecard_pdf_list, ignore_index=True)
# DISPLAY THE EXCESS RETURN GRID (EXIT SIGNAL AND BUY-STOP x COOLING-OFF)
display(exploration_pdf.pivot_table(index=["exit_signal", "buy_stop"], columns="cooling", values="excess_total_return").round(4))
# PREVIEW DATAFRAME
exploration_pdf[["candidate_idx", "exit_signal", "buy_stop", "cooling", "total_return", "buy_hold_total_return", "excess_total_return", "in_market_pct", "exit_count",
                 "buy_stop_count", "recovery_count", "wrong_exit_share", "mean_gain_right", "mean_loss_wrong", "right_side", "wrong_side", "max_drawdown",
                 "annualized_log_excess", "log_excess_ci_low", "log_excess_ci_high"]].round(4)

In [ ]:
"""
Episode scorecard of the slowest candidate of each exit signal (b = 3%, c = 20). S / R > 1 = bought back lower than sold.
"""
# DISPLAY THE TWO SCORECARDS (LARGEST EPISODES FIRST)
scorecard_col_list = ["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "buy_hold_return_out", "max_decline_out_pct", "open_at_end"]
for exit_signal_str in ["trend_ma200", "ath_1pct"]:
    candidate_scorecard_pdf = exploration_scorecard_pdf[(exploration_scorecard_pdf["exit_signal"] == exit_signal_str) & (exploration_scorecard_pdf["buy_stop"] == 0.03) & (exploration_scorecard_pdf["cooling"] == 20)]
    print(f"\n{exit_signal_str}, b = 3%, c = 20:\t{len(candidate_scorecard_pdf)} episodes, {int((candidate_scorecard_pdf['s_over_r'] > 1).sum())} bought back lower, "
          f"product of S/R {np.prod(candidate_scorecard_pdf['s_over_r']):.4f}")
    display(candidate_scorecard_pdf.sort_values("sessions_out", ascending=False)[scorecard_col_list].head(10).round(4))

In [ ]:
# SAVE THE RESULTS
write_csv_file_to_path(exploration_pdf, f"{output_path_str}exploration_candidate_data.csv", "W")
write_csv_file_to_path(exploration_scorecard_pdf, f"{output_path_str}exploration_scorecard_data.csv", "W")

In [ ]:
# LOG THE 12 CANDIDATES AS EXPLORATION TRIALS (ONE WRITE)
exploration_trial_pdf = log_trial_pdf(exp_config, "exploration", exploration_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["exit_signal", "buy_stop", "cooling", "window"],
                                      [col for col in exploration_pdf.columns if col not in ["candidate_idx", "exit_signal", "exit_order", "buy_stop", "cooling"]], False,
                                      note_str_in="exp06 step01 exploration (2005-2014 continuous replay, context only; data-dependent follow-up of exp03)")
print(f"Trials logged:\t{len(exploration_trial_pdf)}")

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)